In [0]:
# CELL 1: INSTALL LIBRARIES
%pip install yfinance

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# CELL 2: IMPORTS AND DATA DOWNLOAD
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Nifty 50 tickers
NIFTY50 = [
    "RELIANCE.NS", "TCS.NS", "HDFCBANK.NS", "INFY.NS",
    "ICICIBANK.NS", "HINDUNILVR.NS", "ITC.NS", "SBIN.NS",
    "BHARTIARTL.NS", "KOTAKBANK.NS", "LT.NS", "AXISBANK.NS",
    "ASIANPAINT.NS", "MARUTI.NS", "TITAN.NS", "SUNPHARMA.NS",
    "ULTRACEMCO.NS", "WIPRO.NS", "HCLTECH.NS", "BAJFINANCE.NS",
    "NESTLEIND.NS", "POWERGRID.NS", "NTPC.NS", "TECHM.NS",
    "HDFCLIFE.NS", "BAJAJFINSV.NS", "DIVISLAB.NS", "DRREDDY.NS",
    "CIPLA.NS", "BRITANNIA.NS", "COALINDIA.NS", "GRASIM.NS",
    "INDUSINDBK.NS", "JSWSTEEL.NS", "TATAMOTORS.NS", "TATACONSUM.NS",
    "TATASTEEL.NS", "ADANIENT.NS", "ADANIPORTS.NS", "APOLLOHOSP.NS",
    "BAJAJ-AUTO.NS", "BPCL.NS", "EICHERMOT.NS", "HEROMOTOCO.NS",
    "HINDALCO.NS", "M&M.NS", "ONGC.NS", "SBILIFE.NS",
    "SHRIRAMFIN.NS", "UPL.NS"
]

print("Downloading Nifty 50 data...")
raw = yf.download(NIFTY50, start="2019-01-01",
                  end="2024-12-31", auto_adjust=True, progress=True)

prices = raw['Close']

# Clean — remove stocks with >10% missing
missing = prices.isnull().mean()
prices  = prices[missing[missing <= 0.1].index]
prices  = prices.dropna(how='all')

# Download index
index_raw   = yf.download("^NSEI", start="2019-01-01",
                           end="2024-12-31", auto_adjust=True, progress=False)
index_price = index_raw['Close'].squeeze()

print(f"✓ Prices shape : {prices.shape}")
print(f"✓ Index points : {len(index_price)}")

[******                12%                       ]  6 of 50 completedHTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: TATAMOTORS.NS"}}}
$TATAMOTORS.NS: possibly delisted; no timezone found
[*********************100%***********************]  49 of 50 completed

1 Failed download:
['TATAMOTORS.NS']: possibly delisted; no timezone found


✓ Prices shape : (1480, 49)
✓ Index points : 1477


In [0]:
# CELL 3: SAVE TO DELTA LAKE
prices_reset = prices.reset_index()
prices_reset.columns = [str(c) for c in prices_reset.columns]

prices_spark = spark.createDataFrame(prices_reset)


prices_spark.write.format("delta") \
            .mode("overwrite") \
            .saveAsTable("nifty50_prices")

print("Prices saved to Delta Lake as 'nifty50_prices'")

index_df = index_price.reset_index()
index_df.columns = ['Date', 'Close']
index_spark = spark.createDataFrame(index_df)
index_spark.write.format("delta") \
           .mode("overwrite") \
           .saveAsTable("nifty50_index")

print("Index saved to Delta Lake as 'nifty50_index'")
print("\nThese tables now live permanently in the cloud. You never need to download this data again.")

Prices saved to Delta Lake as 'nifty50_prices'
Index saved to Delta Lake as 'nifty50_index'

These tables now live permanently in the cloud. You never need to download this data again.


In [0]:
# CELL 4: READ BACK FROM DELTA LAKE
prices_from_cloud = spark.table("nifty50_prices").toPandas()
prices_from_cloud = prices_from_cloud.set_index('Date').sort_index()

index_from_cloud  = spark.table("nifty50_index").toPandas()
index_from_cloud  = index_from_cloud.set_index('Date').sort_index()

print("Data read back from Delta Lake successfully")
print(f" Prices shape : {prices_from_cloud.shape}")
print(f" Index points : {len(index_from_cloud)}")
print(f"\nFirst 3 rows from cloud:")
print(prices_from_cloud.iloc[:3, :3].round(2))

Data read back from Delta Lake successfully
 Prices shape : (1480, 49)
 Index points : 1477

First 3 rows from cloud:
            ADANIENT.NS  ADANIPORTS.NS  APOLLOHOSP.NS
Date                                                 
2019-01-01       155.31         373.29        1240.77
2019-01-02       152.94         365.90        1233.28
2019-01-03       150.62         362.83        1241.02


In [0]:
# CELL 5: FACTOR ANALYSIS ON CLOUD DATA 

prices = prices_from_cloud.copy()
index_price = index_from_cloud['Close'].copy()

# Daily returns
daily_returns = prices.pct_change().dropna(how='all')

# Factor 1: Momentum
price_1m_ago  = prices.shift(21)
price_12m_ago = prices.shift(252)
momentum      = (price_1m_ago / price_12m_ago) - 1

# Factor 2: Low Volatility 
low_vol_factor = daily_returns.rolling(126).std() * -1

# Factor 3: Mean Reversion 
mean_reversion_factor = prices.pct_change(21) * -1

# Standardise and Combine 
def standardise(factor):
    mean = factor.mean(axis=1)
    std  = factor.std(axis=1)
    return factor.sub(mean, axis=0).div(std, axis=0)

momentum_z   = standardise(momentum)
low_vol_z    = standardise(low_vol_factor)
mean_rev_z   = standardise(mean_reversion_factor)

combined_score = (momentum_z + low_vol_z + mean_rev_z) / 3

# Latest rankings
latest = combined_score.iloc[-1].dropna()\
         .sort_values(ascending=False)

print("=" * 45)
print("  FACTOR RANKINGS — Databricks Cloud")
print("  Data source: Delta Lake (nifty50_prices)")
print("=" * 45)
print(f"\nTop 5 stocks:")
print(latest.head(5).round(3))
print(f"\nBottom 5 stocks:")
print(latest.tail(5).round(3))
print(f"\nFull pipeline running on Databricks cloud")
print(f"Data source: Delta Lake — not local files")

  FACTOR RANKINGS — Databricks Cloud
  Data source: Delta Lake (nifty50_prices)

Top 5 stocks:
POWERGRID.NS     0.987
HEROMOTOCO.NS    0.981
BHARTIARTL.NS    0.625
SBIN.NS          0.552
NTPC.NS          0.501
Name: 2024-12-30 00:00:00, dtype: float64

Bottom 5 stocks:
BAJFINANCE.NS   -0.792
DRREDDY.NS      -0.835
ADANIPORTS.NS   -1.120
INDUSINDBK.NS   -1.155
ADANIENT.NS     -2.311
Name: 2024-12-30 00:00:00, dtype: float64

Full pipeline running on Databricks cloud
Data source: Delta Lake — not local files
